# Lab Assignment 1: Text Processing Techniques
**Author:** Kyuhyeon "David" Cho

**ASU ID:** 1237610792

**Date:** February 1, 2026

In [43]:
# Code Cell 1 - Import necessary libraries
# Library and data import (use the first 1000 rows;
# adding hyperparameter "nrows = 1000" to pd.read_csv funcation).
# Show the summary of the input data.
import pandas as pd
import spacy
from collections import Counter
from google.colab import drive

# Mount Google Drive
# This will trigger an authorization prompt
drive.mount('/content/drive')

# Load the spaCy English model
# In Colab, we often need to download it first just in case
!python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# Requirement: Load the dataset (LIMIT 1000 ROWS)
# Assuming the file is directly in your "My Drive" folder
file_path = '/content/drive/MyDrive/restaurant_reviews_az.csv'

try:
    df = pd.read_csv(file_path, nrows=1000)
    print(" File loaded successfully!")
except FileNotFoundError:
    print(" File not found. Please check that 'restaurant_reviews_az.csv' is in your main Google Drive folder.")

# Requirement: Show the summary of the input data
print("\nData Info:")
print(df.info())
print("\nFirst 5 Rows:")
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
 File loaded successfully!

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    1000 non-null   object
 1   user_id      1000 non-null   object
 2   business_id  1000 non-null   object
 3   stars        1000 non-null   int64 
 4   useful       1000 non-null   int64 
 5   funny        1000

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,Sentiment
0,IVS7do_HBzroiCiymNdxDg,fdFgZQQYQJeEAshH4lxSfQ,sGy67CpJctjeCWClWqonjA,3,1,1,0,"OK, the hype about having Hatch chili in your ...",1/27/2020 22:59,1
1,QP2pSzSqpJTMWOCuUuyXkQ,JBLWSXBTKFvJYYiM-FnCOQ,3w7NRntdQ9h0KwDsksIt5Q,5,1,1,1,Pandemic pit stop to have an ice cream.... onl...,4/19/2020 5:33,1
2,oK0cGYStgDOusZKz9B1qug,2_9fKnXChUjC5xArfF8BLg,OMnPtRGmbY8qH_wIILfYKA,5,1,0,0,I was lucky enough to go to the soft opening a...,2/29/2020 19:43,1
3,E_ABvFCNVLbfOgRg3Pv1KQ,9MExTQ76GSKhxSWnTS901g,V9XlikTxq0My4gE8LULsjw,5,0,0,0,I've gone to claim Jumpers all over the US and...,3/14/2020 21:47,1
4,Rd222CrrnXkXukR2iWj69g,LPxuausjvDN88uPr-Q4cQA,CA5BOxKRDPGJgdUQ8OUOpw,4,1,0,0,"If you haven't been to Maynard's kitchen, it'...",1/17/2020 20:32,1


In [44]:
# Code Cell 2 - Select the 1 star reviews and 5 star reviews from the dataset.
# Requirement: Select the 1 star reviews
one_star_reviews = df[df['stars'] == 1]

# Requirement: Select the 5 star reviews
five_star_reviews = df[df['stars'] == 5]

# Validation print (Good practice for verifying data counts)
print(f"Number of 1-Star Reviews: {len(one_star_reviews)}")
print(f"Number of 5-Star Reviews: {len(five_star_reviews)}")

# Validation: Display first 2 rows of each to prove selection
print("\nSample 1-Star:")
display(one_star_reviews.head(2))

print("\nSample 5-Star:")
display(five_star_reviews.head(2))

Number of 1-Star Reviews: 132
Number of 5-Star Reviews: 562

Sample 1-Star:


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,Sentiment
5,kx6O_lyLzUnA7Xip5wh2NA,YsINprB2G1DM8qG1hbrPUg,rViAhfKLKmwbhTKROM9m0w,1,0,0,0,I stay at the Main Hotel at the Casino from Ju...,7/14/2020 16:43,0
13,PKlfSydanGs_EAeRFhpxNA,YCDZfgCTE1qSby-NEVqyMg,UCMSWPqzXjd7QHq7v8PJjQ,1,2,0,0,I don't know what happened to this place but e...,1/25/2020 22:59,0



Sample 5-Star:


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,Sentiment
1,QP2pSzSqpJTMWOCuUuyXkQ,JBLWSXBTKFvJYYiM-FnCOQ,3w7NRntdQ9h0KwDsksIt5Q,5,1,1,1,Pandemic pit stop to have an ice cream.... onl...,4/19/2020 5:33,1
2,oK0cGYStgDOusZKz9B1qug,2_9fKnXChUjC5xArfF8BLg,OMnPtRGmbY8qH_wIILfYKA,5,1,0,0,I was lucky enough to go to the soft opening a...,2/29/2020 19:43,1


In [45]:
# Code Cell 3 - Apply necessary text processing techniques
# (i.e., segmentation/tokenization, pos tagging, lemmatization, NER, parsing on the selected reviews).
# Use either Spacy or NLTK – one toolkit can suffice.

# Helper function to process text using spaCy pipeline
# This handles Tokenization, POS Tagging, Lemmatization, and NER all at once
def process_reviews(review_series):
    # nlp.pipe is more efficient than a loop for processing many documents
    docs = list(nlp.pipe(review_series))
    return docs

# Requirement: Apply processing to 1-star reviews
docs_1_star = process_reviews(one_star_reviews['text'].astype(str))

# Requirement: Apply processing to 5-star reviews
docs_5_star = process_reviews(five_star_reviews['text'].astype(str))

print("Text processing complete.")

# Verification: Show the first processed review to prove all techniques were applied
print("Sample processed review:")
sample_doc = docs_1_star[0]
for token in sample_doc[:5]: # Just show first 5 tokens
    print(f"Token: {token.text}, Lemma: {token.lemma_}, POS: {token.pos_}, Dep: {token.dep_}, Entity: {token.ent_type_}")

Text processing complete.
Sample processed review:
Token: I, Lemma: I, POS: PRON, Dep: nsubj, Entity: 
Token: stay, Lemma: stay, POS: VERB, Dep: ccomp, Entity: 
Token: at, Lemma: at, POS: ADP, Dep: prep, Entity: 
Token: the, Lemma: the, POS: DET, Dep: det, Entity: FAC
Token: Main, Lemma: Main, POS: PROPN, Dep: compound, Entity: FAC


In [46]:
# Code Cell 4 - Find the top 20 frequently used nouns in 1 star reviews and 5 star reviews, respectively.
# Function to extract specific POS tags
def get_top_pos(docs, pos_tag, top_n=20):
    tokens = []
    for doc in docs:
        # We use token.lemma_ to get the base form (e.g., "dishes" -> "dish")
        # We filter out stop words and punctuation for cleaner results
        tokens.extend([token.lemma_.lower() for token in doc if token.pos_ == pos_tag and not token.is_stop and not token.is_punct])
    return Counter(tokens).most_common(top_n)

# Requirement: Top 20 Nouns for 1-star reviews
print("Top 20 Nouns (1-Star):", get_top_pos(docs_1_star, 'NOUN'))

# Requirement: Top 20 Nouns for 5-star reviews
print("Top 20 Nouns (5-Star):", get_top_pos(docs_5_star, 'NOUN'))

Top 20 Nouns (1-Star): [('food', 90), ('order', 67), ('time', 61), ('service', 48), ('minute', 44), ('place', 43), ('restaurant', 32), ('customer', 29), ('people', 26), ('location', 24), ('pizza', 20), ('hour', 18), ('table', 18), ('wing', 17), ('drive', 17), ('manager', 16), ('chicken', 16), ('employee', 15), ('phone', 14), ('fry', 14)]
Top 20 Nouns (5-Star): [('food', 316), ('place', 236), ('time', 171), ('service', 164), ('restaurant', 103), ('pizza', 89), ('staff', 72), ('order', 65), ('flavor', 62), ('drink', 57), ('chicken', 53), ('meal', 52), ('dish', 50), ('menu', 48), ('salad', 44), ('experience', 44), ('breakfast', 43), ('day', 43), ('sauce', 41), ('beer', 39)]


In [47]:
# Code Cell 5 - Find the top 20 frequently used adjectives in 1 star reviews and 5 star, respectively.
# Requirement: Top 20 Adjectives for 1-star reviews
print("Top 20 Adjectives (1-Star):", get_top_pos(docs_1_star, 'ADJ'))

# Requirement: Top 20 Adjectives for 5-star reviews
print("Top 20 Adjectives (5-Star):", get_top_pos(docs_5_star, 'ADJ'))

Top 20 Adjectives (1-Star): [('bad', 43), ('good', 30), ('rude', 16), ('terrible', 14), ('cold', 12), ('horrible', 11), ('wrong', 10), ('great', 10), ('sad', 10), ('old', 10), ('well', 9), ('little', 9), ('big', 8), ('french', 8), ('sure', 8), ('right', 8), ('busy', 8), ('awful', 7), ('single', 7), ('small', 7)]
Top 20 Adjectives (5-Star): [('good', 319), ('great', 280), ('amazing', 149), ('delicious', 139), ('friendly', 97), ('nice', 77), ('fresh', 55), ('favorite', 53), ('perfect', 52), ('excellent', 46), ('hot', 38), ('wonderful', 37), ('happy', 34), ('new', 33), ('sure', 32), ('awesome', 31), ('big', 30), ('sweet', 30), ('mexican', 29), ('small', 29)]


In [48]:
# Code Cell 6 - Find the top 20 frequently used verbs in 1 star reviews and 5 star reviews, respectively.
# Requirement: Top 20 Verbs for 1-star reviews
print("Top 20 Verbs (1-Star):", get_top_pos(docs_1_star, 'VERB'))

# Requirement: Top 20 Verbs for 5-star reviews
print("Top 20 Verbs (5-Star):", get_top_pos(docs_5_star, 'VERB'))

Top 20 Verbs (1-Star): [('order', 57), ('say', 52), ('go', 40), ('tell', 40), ('wait', 38), ('come', 36), ('get', 35), ('try', 35), ('ask', 30), ('call', 26), ('leave', 23), ('take', 23), ('want', 21), ('eat', 19), ('know', 15), ('charge', 15), ('taste', 14), ('give', 14), ('place', 13), ('think', 12)]
Top 20 Verbs (5-Star): [('come', 151), ('love', 137), ('try', 124), ('order', 122), ('get', 102), ('eat', 90), ('go', 86), ('recommend', 73), ('look', 65), ('wait', 47), ('want', 45), ('enjoy', 42), ('take', 37), ('stop', 35), ('visit', 35), ('know', 34), ('thank', 33), ('cook', 31), ('like', 31), ('find', 30)]


In [49]:
# Code Cell 7 - Find the top 20 frequently used named entities from the selected reviews.

def get_top_entities(docs, top_n=20):
    entities = []
    for doc in docs:
        # ent.text gives the entity string
        # We perform a quick check to filter out empty/whitespace entities
        for ent in doc.ents:
            if ent.text.strip():
                entities.append(ent.text.strip())

    return Counter(entities).most_common(top_n)

# Requirement: Top 20 entities from 1-star reviews
print("Top 20 Named Entities (1-Star):", get_top_entities(docs_1_star))

# Requirement: Top 20 entities from 5-star reviews
print("Top 20 Named Entities (5-Star):", get_top_entities(docs_5_star))

Top 20 Named Entities (1-Star): [('two', 13), ('Taco Bell', 11), ('one', 10), ('2', 10), ('Tucson', 8), ('3', 8), ('first', 7), ('20', 7), ('today', 6), ('French', 5), ('10', 5), ('Mexican', 5), ('Waffle House', 5), ('McDonalds', 5), ('1', 4), ('McDonald', 4), ('zero', 4), ('15 minutes', 4), ('10 minutes', 4), ('french', 3)]
Top 20 Named Entities (5-Star): [('Tucson', 110), ('first', 34), ('Mexican', 26), ('one', 22), ('two', 22), ('French', 20), ('5', 18), ('Arizona', 16), ('2', 15), ('3', 12), ('First', 12), ('Persian', 12), ('today', 10), ('10/10', 9), ('hour', 9), ('Sunday', 8), ('One', 8), ('Sonoran', 8), ('Yelp', 8), ('the next day', 7)]


## Tect Cell 8
- I utilized Google Gemini to assist in debugging the spaCy processing pipeline and structuring the pandas dataframe operations.

## Text Cell 9
- Observation: When analyzing the frequently used nouns, I found that both 1-star and 5-star reviews focus on the same core topics: "food," "time," and "service." The real difference appears in the adjectives. 1-star reviews frequently use negative descriptors like "bad," "rude," "terrible," and "cold," which point to specific failures in service attitude and food temperature. In contrast, 5-star reviews are dominated by positive adjectives like "good," "great," "amazing," "delicious," and "friendly."

- Key to a Good Restaurant Experience: The key to a good experience is the ability to generate positive sentiment across both food and service. Since the nouns are the same in both groups, a restaurant's success depends on execution. A 5-star experience is defined by converting those neutral topics into positive emotional connections—making the food "delicious" instead of "cold," and the service "friendly" instead of "rude."

## Text Cell 10

### 1. The Prompt Input used for the GenAI Model:

"I performed a text analysis on Arizona restaurant reviews and found that the most frequent nouns in both 1-star and 5-star reviews are identical ('food', 'service', 'time'). However, the adjectives are opposites: 1-star reviews feature 'rude', 'cold', and 'terrible', while 5-star reviews feature 'friendly', 'delicious', and 'amazing'. Based on this data, what is the key differentiator for a good restaurant experience?"

### 2. The GenAI Response Below is the response generated by the AI:

"The fact that the nouns are identical suggests that customer expectations are consistent: everyone cares about food, service, and speed. The differentiator is purely execution.

- Service Recovery vs. Failure: The presence of 'rude' in 1-star reviews implies that it wasn't just a mistake, but a lack of empathy from the staff that sealed the negative fate.

- Temperature & Quality: The contrast between 'cold' and 'delicious' shows that basics matter. A 5-star experience isn't necessarily about luxury; it is about hitting the standard (hot food, warm service) consistently. The key to a good experience is transforming those neutral nouns into positive emotional touchpoints."

### 3. Commentary & Justification Comparison of Manual Analysis vs. GenAI:
My manual analysis was strictly data-driven: I looked at the specific word counts and noticed the overlap in nouns versus the divergence in adjectives. The GenAI took those findings and provided a "root cause" explanation, suggesting that "execution" and "empathy" are the reasons behind the words.

### Refinement & Final Optimal Version:

The AI's response was satisfactory but theoretical. My original observation was accurate but lacked the "why." The optimal version combines my data evidence with the AI's reasoning:

"The key to a good restaurant experience is execution. My data proves that customers judge all restaurants on the same three metrics: food, service, and time. A restaurant succeeds not by offering something new, but by ensuring those specific metrics are described positively (e.g., ensuring food is 'hot' rather than 'cold'). The difference between a 1-star and 5-star rating is simply the gap between expectation and delivery on these core basics."

In [50]:
import os
import subprocess

# 1. Search for your specific file in Google Drive
print("🔍 Searching for 'LA1_Cho_Kyuhyeon.ipynb' in your Drive...")
found_files = subprocess.getoutput("find /content/drive -name 'LA1_Cho_Kyuhyeon.ipynb'").split('\n')

if found_files and found_files[0]:
    file_path = found_files[0]
    print(f"✅ Found file at: {file_path}")

    # 2. Convert the file using the path we just found
    print("⏳ Converting to HTML...")
    cmd = f"jupyter nbconvert '{file_path}' --to html --output-dir='/content'"
    os.system(cmd)

    print("\n🎉 SUCCESS! The HTML file is now in the sidebar folder.")
    print("👉 Click the Folder Icon (left) -> Refresh -> Right-click 'LA1_Cho_Kyuhyeon.html' -> Download")
else:
    print("❌ Could not find the file. Please ensure it is named EXACTLY 'LA1_Cho_Kyuhyeon.ipynb' in your Google Drive.")

🔍 Searching for 'LA1_Cho_Kyuhyeon.ipynb' in your Drive...
✅ Found file at: /content/drive/MyDrive/Colab Notebooks/LA1_Cho_Kyuhyeon.ipynb
⏳ Converting to HTML...

🎉 SUCCESS! The HTML file is now in the sidebar folder.
👉 Click the Folder Icon (left) -> Refresh -> Right-click 'LA1_Cho_Kyuhyeon.html' -> Download


In [51]:
from google.colab import files

# This will trigger the browser to download the file directly
files.download('LA1_Cho_Kyuhyeon.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>